# Análisis de Capturas Pesqueras mediante Modelos de Machine Learning

In [1]:
#INSTALACIÓN DE DEPENDENCIAS
!pip install -q xgboost statsmodels scikit-learn pandas matplotlib seaborn plotly umap-learn

/bin/bash: línea 1: pip: orden no encontrada


In [2]:
# IMPORTS Y CONFIGURACIÓN GENERAL
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

# XGBoost
from xgboost import XGBRegressor

# Series de tiempo
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Configuración visual
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

# Semilla para reproducibilidad
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Librerías cargadas correctamente")

ModuleNotFoundError: No module named 'seaborn'

# FASE 1 — COMPRENSIÓN DEL NEGOCIO

**Problema:** Los organismos de gestión pesquera en Argentina carecen de
herramientas predictivas para estimar volúmenes de captura por especie,
puerto y flota, lo que genera incertidumbre en la logística portuaria
y la asignación de recursos operativos.

**Objetivo medible:** Desarrollar modelos predictivos que estimen la
captura mensual (en kg) a nivel de registro (especie × puerto × flota × mes),
reduciendo al menos un 15% el MAE respecto del baseline naïve (captura del
mismo mes del año anterior para el mismo grupo).

**Estrategia de modelado:**
- **Modelo global:** un único modelo con todos los registros.
- **Modelos segmentados por especie agrupada:** un modelo independiente
  por cada especie del Top 5 por volumen.
- **Comparación:** ambos enfoques evaluados sobre el mismo test (2019).

*Justificación de segmentar por especie:* cada especie tiene ciclos
biológicos y estacionalidades propias. Se validará en el EDA que los
perfiles mensuales difieren significativamente entre especies.

**KPIs del modelo:**

| Métrica | Umbral de éxito |
|---------|----------------|
| MAE     | Reducción ≥ 15% vs. baseline |
| MAPE    | < 50% |
| R²      | > 0.5 |

**Hipótesis:**
1. Existen patrones estacionales diferenciados por especie → validar con
   perfil mensual y test de Kruskal-Wallis
2. La ubicación geográfica influye en el volumen → validar con importancia
   de features (lat/lon)
3. El tipo de flota impacta la captura → validar con importancia de
   features (flota)

**Restricciones:**
- Período: 2010-01 a 2019-11 (119 meses; falta diciembre 2019)
- Sin variables exógenas (clima, cuotas, precios)
- 5.85% de coordenadas geográficas nulas
- Series irregulares por grupo (no todos tienen registros mensuales continuos)
- Target muy asimétrico: media ~170K kg vs. mediana ~2.8K kg

**Validación temporal:**
- Train: 2010–2018 | Test: 2019
- El test set NO se usa para ajustar hiperparámetros ni early stopping

In [ ]:
df = pd.read_csv('data/captura-puerto-flota-2010-2019.csv')

print(f"Dimensiones del dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Columnas: {list(df.columns)}")

# Verificación básica de integridad
print(f"\nRegistros duplicados: {df.duplicated().sum():,}")
print(f"Rango de fechas: {df['fecha'].min()} a {df['fecha'].max()}")

# Cada fila = una combinación de especie × puerto × flota × mes
# La variable 'captura' está expresada en kilogramos
df.head(10)

In [ ]:
# INSPECCIÓN INICIAL DEL DATASET
print("=" * 60)
print("INFORMACIÓN GENERAL")
print("=" * 60)
df.info()

# Estadísticas solo de variables numéricas relevantes
print("\n" + "=" * 60)
print("ESTADÍSTICAS DESCRIPTIVAS — VARIABLE OBJETIVO")
print("=" * 60)
stats_captura = df['captura'].describe()
print(stats_captura)
print(f"\nAsimetría (skewness): {df['captura'].skew():.2f}")
print(f"Ratio media/mediana:  {df['captura'].mean() / df['captura'].median():.1f}x")
print(f"Registros con captura = 0: {(df['captura'] == 0).sum()}")
print(f"Registros con captura < 0: {(df['captura'] < 0).sum()}")

# Nota: provincia_id y departamento_id son códigos, no variables numéricas.
# No se incluyen en el análisis descriptivo.

# Coordenadas geográficas
print("\n" + "=" * 60)
print("ESTADÍSTICAS — COORDENADAS GEOGRÁFICAS")
print("=" * 60)
display(df[['latitud', 'longitud']].describe())

# Valores nulos
print("\n" + "=" * 60)
print("VALORES NULOS POR COLUMNA")
print("=" * 60)
nulos = df.isnull().sum()
nulos_pct = (df.isnull().sum() / len(df) * 100).round(2)
tabla_nulos = pd.DataFrame({'Nulos': nulos, '% Nulos': nulos_pct})
display(tabla_nulos[tabla_nulos['Nulos'] > 0])

print(f"\n Las coordenadas nulas corresponden a puertos sin "
      f"georreferenciación ({nulos['latitud']:,} registros, {nulos_pct['latitud']}%)")

In [ ]:
# CARDINALIDAD DE VARIABLES CATEGÓRICAS
categoricas = ['flota', 'puerto', 'provincia', 'categoria', 'especie', 'especie_agrupada']

for col in categoricas:
    n_unique = df[col].nunique()
    print(f"\n{'='*50}")
    print(f"{col.upper()} — {n_unique} valores únicos")
    print(f"{'='*50}")
    print(df[col].value_counts().head(10))

# Resumen para decisiones de modelado
print("\n" + "=" * 60)
print("RESUMEN DE CARDINALIDAD PARA MODELADO")
print("=" * 60)
print(f"""
- especie: 90 valores únicos → Demasiados para segmentar
- especie_agrupada: 16 valores → Adecuada para segmentación
  (se usará esta variable para los modelos por especie)
- flota: 8 valores únicos
- puerto: 23 valores únicos
- provincia: 6 valores (incluye 'sin especificar': {(df['provincia']=='sin especificar').sum()} registros)
- categoria: 4 valores (incluye 'otras': {(df['categoria']=='otras').sum()} registro)
""")

# Top 5 especies agrupadas por volumen (las que se usarán para segmentar)
top5_vol = (df.groupby('especie_agrupada')['captura']
            .sum()
            .sort_values(ascending=False)
            .head(5))

print("TOP 5 ESPECIES AGRUPADAS POR VOLUMEN TOTAL (candidatas para segmentación):")
for i, (esp, vol) in enumerate(top5_vol.items(), 1):
    n_registros = (df['especie_agrupada'] == esp).sum()
    print(f"  {i}. {esp}: {vol/1e6:.1f} M kg ({n_registros:,} registros)")

---
# FASE 2 — COMPRENSIÓN DE LOS DATOS (EDA)
---

In [ ]:
df['fecha'] = pd.to_datetime(df['fecha'], format='%Y-%m')
df['anio'] = df['fecha'].dt.year
df['mes'] = df['fecha'].dt.month
df['trimestre'] = df['fecha'].dt.quarter

# Verificar rango temporal
meses_esperados = 12 * (2019 - 2010 + 1)  # 120 meses si estuviera completo
meses_reales = df['fecha'].dt.to_period('M').nunique()

print(f"Período: {df['fecha'].min().strftime('%Y-%m')} a {df['fecha'].max().strftime('%Y-%m')}")
print(f"Años disponibles: {sorted(df['anio'].unique().tolist())}")
print(f"Meses distintos: {meses_reales} de {meses_esperados} esperados")

if meses_reales < meses_esperados:
    # Identificar qué meses faltan
    rango_completo = pd.period_range('2010-01', '2019-12', freq='M')
    meses_presentes = df['fecha'].dt.to_period('M').unique()
    meses_faltantes = rango_completo.difference(meses_presentes)
    print(f"Meses faltantes: {[str(m) for m in meses_faltantes]}")

# Registros por año (para verificar que 2019 no esté incompleto)
print(f"\nRegistros por año:")
print(df.groupby('anio').size().to_string())

In [ ]:
# EDA — SERIE TEMPORAL TOTAL
serie_mensual = df.groupby('fecha')['captura'].sum().reset_index()
serie_mensual.columns = ['fecha', 'captura_total']

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Serie temporal
axes[0].plot(serie_mensual['fecha'], serie_mensual['captura_total'] / 1e6,
             color='steelblue', linewidth=1.5)
axes[0].set_title('Captura Total Mensual (Argentina, 2010–2019)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Captura (millones de kg)')
axes[0].axhline(serie_mensual['captura_total'].mean() / 1e6, color='red',
                linestyle='--', alpha=0.7, label='Promedio')
axes[0].legend()

# Box plot por mes (estacionalidad)
serie_mensual['mes'] = serie_mensual['fecha'].dt.month
sns.boxplot(data=serie_mensual, x='mes', y='captura_total', ax=axes[1], palette='coolwarm')
axes[1].set_title('Distribución de Captura por Mes (Estacionalidad)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Captura (kg)')
axes[1].set_xlabel('Mes')

plt.tight_layout()
plt.show()

# KPI: Coeficiente de variación
cv = serie_mensual['captura_total'].std() / serie_mensual['captura_total'].mean()
print(f"\n - KPI — Coeficiente de Variación (CV): {cv:.3f}")
print(f" - KPI — Captura mensual promedio: {serie_mensual['captura_total'].mean()/1e6:.2f} M kg")
print(f" - KPI — Captura mensual máxima: {serie_mensual['captura_total'].max()/1e6:.2f} M kg")

In [ ]:
# EDA — SERIE TEMPORAL TOTAL
serie_mensual = df.groupby('fecha')['captura'].sum().reset_index()
serie_mensual.columns = ['fecha', 'captura_total']

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Serie temporal
axes[0].plot(serie_mensual['fecha'], serie_mensual['captura_total'] / 1e6,
             color='steelblue', linewidth=1.5)
axes[0].set_title('Captura Total Mensual (Argentina, 2010–2019)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Captura (millones de kg)')
axes[0].axhline(serie_mensual['captura_total'].mean() / 1e6, color='red',
                linestyle='--', alpha=0.7, label='Promedio')
axes[0].legend()

# Box plot por mes (estacionalidad)
serie_mensual['mes'] = serie_mensual['fecha'].dt.month
sns.boxplot(data=serie_mensual, x='mes', y=serie_mensual['captura_total'] / 1e6,
            ax=axes[1], palette='coolwarm')
axes[1].set_title('Distribución de Captura por Mes (Estacionalidad)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Captura (millones de kg)')
axes[1].set_xlabel('Mes')

plt.tight_layout()
plt.show()

# KPIs de la serie agregada
cv_agregado = serie_mensual['captura_total'].std() / serie_mensual['captura_total'].mean()
cv_individual = df['captura'].std() / df['captura'].mean()

print(f"\nKPIs — Serie Agregada (119 meses):")
print(f"  Captura mensual promedio: {serie_mensual['captura_total'].mean()/1e6:.2f} M kg")
print(f"  Captura mensual máxima:   {serie_mensual['captura_total'].max()/1e6:.2f} M kg")
print(f"  CV serie agregada:        {cv_agregado:.3f} (variabilidad moderada)")
print(f"  CV registros individuales: {cv_individual:.2f} (variabilidad alta)")
print(f"\n⚠️ El CV agregado (0.24) oculta la gran dispersión entre registros")
print(f"   individuales (CV={cv_individual:.2f}), donde media={df['captura'].mean():,.0f} kg")
print(f"   vs mediana={df['captura'].median():,.0f} kg")

In [ ]:
# EDA — CAPTURA POR ESPECIE AGRUPADA
# Top 10 especies por volumen total
top_especies = (df.groupby('especie_agrupada')['captura']
                .sum()
                .sort_values(ascending=False)
                .head(10))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Barras
top_especies.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 10 Especies por Captura Total', fontweight='bold')
axes[0].set_xlabel('Captura Total (kg)')
axes[0].invert_yaxis()

# Evolución temporal top 5
top5 = top_especies.index[:5].tolist()
serie_especie = (df[df['especie_agrupada'].isin(top5)]
                 .groupby(['fecha', 'especie_agrupada'])['captura']
                 .sum()
                 .reset_index())

for esp in top5:
    data = serie_especie[serie_especie['especie_agrupada'] == esp]
    axes[1].plot(data['fecha'], data['captura'] / 1e6, label=esp, linewidth=1.2)

axes[1].set_title('Evolución Mensual — Top 5 Especies', fontweight='bold')
axes[1].set_ylabel('Captura (M kg)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"\n→ Las Top 5 especies ({', '.join(top5)}) concentran la mayor parte")
print(f"  del volumen y muestran dinámicas temporales distintas entre sí.")

# Resumen del Top 5 (candidatas para modelos segmentados)
print("\nTop 5 especies — candidatas para modelos segmentados:")
print("-" * 65)
for i, esp in enumerate(top5, 1):
    datos_esp = df[df['especie_agrupada'] == esp]
    vol_total = datos_esp['captura'].sum()
    n_registros = len(datos_esp)
    n_meses = datos_esp['fecha'].dt.to_period('M').nunique()
    print(f"  {i}. {esp}")
    print(f"     Volumen: {vol_total/1e6:.1f} M kg | "
          f"Registros: {n_registros:,} | "
          f"Meses con datos: {n_meses}/119")

In [ ]:
# EDA — CAPTURA POR FLOTA (validación hipótesis 3)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Total por flota
captura_flota = df.groupby('flota')['captura'].sum().sort_values(ascending=False)
captura_flota.plot(kind='bar', ax=axes[0], color='coral', edgecolor='black')
axes[0].set_title('Captura Total por Tipo de Flota', fontweight='bold')
axes[0].set_ylabel('Captura total (kg)')
axes[0].tick_params(axis='x', rotation=45)

# Evolución temporal por flota
for flota_name in df['flota'].unique():
    data_f = df[df['flota'] == flota_name].groupby('fecha')['captura'].sum()
    axes[1].plot(data_f.index, data_f.values / 1e6, label=flota_name, linewidth=1)

axes[1].set_title('Evolución Mensual por Flota', fontweight='bold')
axes[1].set_ylabel('Captura (M kg)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()
print("\n→ H3: Se observan diferencias en volumen por flota, pero la segmentación")
print("  por especie (H1) tiene mayor justificación biológica. Se usa flota como")
print("  feature del modelo en lugar de como criterio de segmentación.")

In [ ]:
# EDA — ANÁLISIS GEOGRÁFICO (validación hipótesis 2)
# Top 10 puertos
top_puertos = (df.groupby('puerto')['captura']
               .sum()
               .sort_values(ascending=False)
               .head(10))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

top_puertos.plot(kind='barh', ax=axes[0], color='teal')
axes[0].set_title('Top 10 Puertos por Captura Total', fontweight='bold')
axes[0].invert_yaxis()

# Captura por provincia
captura_prov = df.groupby('provincia')['captura'].sum().sort_values(ascending=False)
captura_prov.plot(kind='barh', ax=axes[1], color='darkorange')
axes[1].set_title('Captura Total por Provincia', fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

# Mapa de calor si hay coordenadas
if df['latitud'].notna().sum() > 0:
    fig_map = px.scatter_mapbox(
        df.groupby(['puerto', 'latitud', 'longitud'])['captura'].sum().reset_index(),
        lat='latitud', lon='longitud', size='captura', color='captura',
        hover_name='puerto', zoom=3, height=500,
        color_continuous_scale='YlOrRd',
        title='Distribución Geográfica de Capturas por Puerto'
    )
    fig_map.update_layout(mapbox_style='open-street-map')
    fig_map.show()

print("\n→ H2: La concentración geográfica es evidente (Mar del Plata domina).")
print("  Se incorporan latitud, longitud y puerto como features del modelo.")
print("  No se segmenta por región porque la mayoría del volumen se concentra")
print("  en pocos puertos, lo que dejaría segmentos con datos insuficientes.")

In [ ]:
# DESCOMPOSICIÓN TEMPORAL (Tendencia + Estacionalidad + Residuo)
serie_ts = serie_mensual.set_index('fecha')['captura_total']
serie_ts = serie_ts.asfreq('MS')  # frecuencia mensual

decomposition = seasonal_decompose(serie_ts, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(16, 12))
decomposition.observed.plot(ax=axes[0], title='Serie Original')
decomposition.trend.plot(ax=axes[1], title='Tendencia')
decomposition.seasonal.plot(ax=axes[2], title='Estacionalidad')
decomposition.resid.plot(ax=axes[3], title='Residuo')

for ax in axes:
    ax.set_xlabel('')

plt.suptitle('Descomposición de Serie Temporal — Captura Total Mensual',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Test de estacionariedad (ADF)
adf_result = adfuller(serie_ts.dropna())
print(f"\n - Test Dickey-Fuller Aumentado:")
print(f"   Estadístico ADF: {adf_result[0]:.4f}")
print(f"   p-value: {adf_result[1]:.4f}")
print(f"   {'Serie estacionaria' if adf_result[1] < 0.05 else ' Serie NO estacionaria'}")

print(f"\n→ La serie es estacionaria (p=0.02 < 0.05), con estacionalidad")
print(f"  anual visible (período=12). Esto soporta el uso de SARIMA.")

---
# FASE 3 — PREPARACIÓN DE LOS DATOS
---

In [ ]:
# LIMPIEZA DE DATOS
print("Estado inicial:")
print(f"  Filas: {len(df):,}")
print(f"  Nulos en latitud:  {df['latitud'].isna().sum()}")
print(f"  Nulos en longitud: {df['longitud'].isna().sum()}")
print(f"  Nulos en captura:  {df['captura'].isna().sum()}")

# 1. Imputación de coordenadas por promedio de puerto
for col_geo in ['latitud', 'longitud']:
    media_por_puerto = df.groupby('puerto')[col_geo].transform('mean')
    df[col_geo] = df[col_geo].fillna(media_por_puerto)

# 2. Imputar los restantes con mediana general
#    (puertos donde TODOS los registros tenían NaN)
nulos_restantes = df['latitud'].isna().sum()
if nulos_restantes > 0:
    print(f"\n {nulos_restantes} registros sin coordenadas tras imputación por puerto.")
    print("  Se imputan con la mediana general de lat/lon.")
    df['latitud'] = df['latitud'].fillna(df['latitud'].median())
    df['longitud'] = df['longitud'].fillna(df['longitud'].median())

# 3. Eliminar filas con captura <= 0
filas_antes = len(df)
df = df[df['captura'] > 0].copy()
print(f"\n  Filas eliminadas con captura <= 0: {filas_antes - len(df)}")

# 4. Verificación final
print(f"\nEstado final:")
print(f"  Filas: {len(df):,}")
print(f"  Nulos restantes: {df.isnull().sum().sum()}")

In [ ]:
# INGENIERÍA DE VARIABLES

# 1. Variables cíclicas del mes
df['mes_sin'] = np.sin(2 * np.pi * df['mes'] / 12)
df['mes_cos'] = np.cos(2 * np.pi * df['mes'] / 12)

# 2. Transformación logarítmica del target
df['log_captura'] = np.log1p(df['captura'])

# 3. Lag de 12 meses (CORREGIDO: por fecha, no por posición de fila)
df['fecha_lag12'] = df['fecha'] - pd.DateOffset(months=12)

lag_df = df[['especie_agrupada', 'flota', 'puerto', 'fecha', 'captura']].copy()
lag_df = lag_df.rename(columns={'captura': 'lag_12', 'fecha': 'fecha_lag12'})

df = df.merge(
    lag_df,
    on=['especie_agrupada', 'flota', 'puerto', 'fecha_lag12'],
    how='left'
)
df = df.drop(columns=['fecha_lag12'])

# 4. Media móvil 12 meses (CORREGIDO: excluye valor actual con shift(1))
df = df.sort_values(['especie_agrupada', 'flota', 'puerto', 'fecha'])
df['ma_12_grupo'] = (df.groupby(['especie_agrupada', 'flota', 'puerto'])['captura']
                     .transform(lambda x: x.shift(1).rolling(12, min_periods=1).mean()))

# 5. Codificación de variables categóricas
label_encoders = {}
cat_cols = ['flota', 'puerto', 'provincia', 'categoria', 'especie', 'especie_agrupada']

for col in cat_cols:
    le = LabelEncoder()
    df[f'{col}_enc'] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

# Verificación
print("Feature Engineering completado")
print(f"  Columnas totales: {len(df.columns)}")
print(f"  Registros con lag_12 disponible: {df['lag_12'].notna().sum():,} de {len(df):,}")
print(f"  Registros sin lag_12 (se eliminarán): {df['lag_12'].isna().sum():,}")